<a href="https://colab.research.google.com/github/guuhh/AOC-2019.2/blob/master/Spark_estudos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark import SparkContext
spark_context = SparkContext()
print(spark_context)
print(spark_context.version)
spark_context.stop()

<SparkContext master=local[*] appName=pyspark-shell>
4.0.4


In [ ]:
from pyspark.sql import SparkSession
#spark_session = SparkSession.builder.getOrCreate()
spark_session = SparkSession.builder.master("local").appName("Word Count").getOrCreate()
print(spark_session)
dataset = spark_session.read.csv('/content/sample_data/california_housing_train.csv',
                                 inferSchema=True, header=True)
#dataset.head()
#dataset.count()
#dataset.printSchema()
#dataset.show()
#spark_session.stop()

# create a temporary table and list it
dataset.createOrReplaceTempView('temporary_table')
print(spark_session.catalog.listTables())
#O próximo passo consiste em fazer uma consulta SQL.
#Para isso, vamos selecionar apenas três registros com os dados referentes às colunas “longitude” e “latitude”, executando o seguinte código:
query = 'FROM temporary_table SELECT longitude, latitude LIMIT 3'
#instacie outputquery
output_query = spark_session.sql(query)
output_query.show()
#spark_session.stop()

[Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
+---------+--------+
|longitude|latitude|
+---------+--------+
|  -114.31|   34.19|
|  -114.47|    34.4|
|  -114.56|   33.69|
+---------+--------+



In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

spark_session = SparkSession.builder.master("local").appName("Max_qty_rooms").getOrCreate()
dataset = spark_session.read.csv(
    '/content/sample_data/california_housing_train.csv',
    inferSchema=True,
    header=True)
dataset.createOrReplaceTempView("temporary_table")
print(spark_session.catalog.listTables())

query = 'SELECT MAX(total_rooms) as maximo_quartos FROM temporary_table'
q_maximo_quartos = spark_session.sql(query)
q_maximo_quartos.show()
maximo = q_maximo_quartos.collect()[0]['maximo_quartos']
print('Resultado do spark SQL:',maximo)
pd_maximo_quartos = q_maximo_quartos.toPandas()
print('Quantidade maxima de quartos no Pandas é :{}'.format(pd_maximo_quartos['maximo_quartos']))
print('Quantidade maxima de quartos no Pandas é - apenas imprimir primeira linha :{}'.format(pd_maximo_quartos['maximo_quartos'].iloc[0]))
print('Quantidade maxima de quartos no Pandas é - apenas imprimir primeira e segunda linha :{}'.format(pd_maximo_quartos['maximo_quartos'].iloc[0:2]))
qty_maximo_quartos = int(pd_maximo_quartos.loc[0,'maximo_quartos'])
print('Resultado do Pandas:',qty_maximo_quartos)

query2 = 'SELECT longitude, latitude FROM temporary_table WHERE total_rooms = '+str(qty_maximo_quartos)
localizacao_maximo_quartos = spark_session.sql(query2)
pd_localizacao_maximo_quatos = localizacao_maximo_quartos.toPandas()
print(pd_localizacao_maximo_quatos.head())

media = 0
desvio_padrao = 0.1
pd_temporario = pd.DataFrame(np.random.normal(media, desvio_padrao, 100))
spark_temporario =  spark_session.createDataFrame(pd_temporario)

print(spark_session.catalog.listTables())
spark_temporario.createOrReplaceTempView('new_temporary_table')
print(spark_session.catalog.listTables())
spark_session.stop()



[Table(name='new_temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
+--------------+
|maximo_quartos|
+--------------+
|       37937.0|
+--------------+

Resultado do spark SQL: 37937.0
Quantidade maxima de quartos no Pandas é :0    37937.0
Name: maximo_quartos, dtype: float64
Quantidade maxima de quartos no Pandas é - apenas imprimir primeira linha :37937.0
Quantidade maxima de quartos no Pandas é - apenas imprimir primeira e segunda linha :0    37937.0
Name: maximo_quartos, dtype: float64
Resultado do Pandas: 37937
   longitude  latitude
0    -117.74     33.89
[Table(name='new_temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='temporary_table', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]
[Table(name='new_